## Multinomial Naive Bayes

Suppose we want to build an email spam detector using Multinomial Naive Bayes.

After preprocessing, the vocabulary is: dear, friend, lunch, money.

Training data: 8 ham emails and 4 spam emails (12 total).

For ham emails (total word count = 17):

$$
P(\text{dear} \mid H) = \frac{8}{17} \approx 0.47
$$

$$
P(\text{friend} \mid H) = \frac{5}{17} \approx 0.29
$$

$$
P(\text{lunch} \mid H) = \frac{3}{17} \approx 0.18
$$

$$
P(\text{money} \mid H) = \frac{1}{17} \approx 0.06
$$

For spam emails (total word count = 7):

$$
P(\text{dear} \mid S) = \frac{2}{7} \approx 0.29
$$

$$
P(\text{friend} \mid S) = \frac{1}{7} \approx 0.14
$$

$$
P(\text{lunch} \mid S) = \frac{0}{7} = 0
$$

$$
P(\text{money} \mid S) = \frac{4}{7} \approx 0.57
$$

Class priors:

$$
P(H) = \frac{8}{12} \approx 0.67, \quad P(S) = \frac{4}{12} \approx 0.33
$$

Now consider the new message: "dear friend"

Ham score:

$$
P(H)\,P(\text{dear} \mid H)\,P(\text{friend} \mid H)
= 0.67 \times 0.47 \times 0.29
\approx 0.09
$$

Spam score:

$$
P(S)\,P(\text{dear} \mid S)\,P(\text{friend} \mid S)
= 0.33 \times 0.29 \times 0.14
\approx 0.01
$$

Since $0.09 > 0.01$, the message is classified as ham.

Note: $P(\text{dear} \mid S)$ is the probability of observing the word "dear" given that the email is spam.

Another example (Multinomial Naive Bayes):

New message: "Lunch money money money money"

$$
\text{Score}(H) = P(H)\,P(\text{lunch} \mid H)\,P(\text{money} \mid H)^4 = 0.00002
$$

$$
\text{Score}(S) = P(S)\,P(\text{lunch} \mid S)\,P(\text{money} \mid S)^4 = 0
$$

This shows the zero-probability problem: if one conditional probability is zero, the full product becomes zero.

To fix this, we apply smoothing with $\alpha$, which adds a small count to each word before recomputing probabilities.

Why is Naive Bayes called "naive"?

Because it assumes word independence: the model ignores word order and treats a message as a bag of words.

Even with this simplified assumption, it often performs surprisingly well for spam filtering.

-------------------------------------------------

## Logistic Regression

Logistic Regression is a supervised machine learning algorithm used for classification problems.
Unlike linear regression, which predicts continuous values, it predicts the probability that an input belongs to a specific class.

It is commonly used for binary classification, where the output has two possible categories such as Yes/No, True/False, or 0/1.

The sigmoid function converts the model output into a probability between 0 and 1:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Types of logistic regression:

- Binomial Logistic Regression: Used when the dependent variable has only two categories. Examples include Yes/No, Pass/Fail, or 0/1.
- Multinomial Logistic Regression: Used when the dependent variable has three or more unordered categories. For example, classifying animals as cat, dog, or sheep.
- Ordinal Logistic Regression: Used when the dependent variable has three or more categories with a natural order, such as low, medium, and high.

-------------------------------------------------

## Model Accuracy

Accuracy measures the overall correctness of a model:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

It works well when the classes are balanced, but it can be misleading when the classes are imbalanced.
For example, if 99% of emails are "not spam," a model that always predicts "not spam" can still get 99% accuracy and be useless.

Accuracy is also not enough when false positives or false negatives have different costs, such as in spam filtering or medical tests.

Precision: how many predicted positives are actually positive.

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

High precision means that when the model predicts "spam," it is usually correct.

Recall: how many actual positives are found by the model.

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

High recall means that the model catches most positive cases, such as most cancer cases in a medical test.

F1-score: the harmonic mean of precision and recall.

$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

$$
F1 = \frac{2TP}{2TP + FP + FN}
$$

The F1-score is useful when you want a single measure that balances precision and recall.

----------------------------

In [6]:
import os
os.getcwd()

'D:\\NLP2026\\NLP-2026\\sessions\\session5'

In [2]:
import re
import string
from nltk.corpus import stopwords
from textblob import Word  # or nltk.stem.WordNetLemmatizer
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()  # if using NLTK

def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    text = " ".join(word for word in text.split() if word not in stop_words)
    st = PorterStemmer()
    # text = " ".join(st.stem(word) for word in text.split())
    text = " ".join(Word(word).lemmatize() for word in text.split())
    return text

In [4]:
import pandas as pd
import string
from nltk.corpus import stopwords
from textblob import Word
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import naive_bayes, linear_model, metrics
import joblib

# 1. Load and prepare data
df = pd.read_csv("spamHam.tsv", delimiter='\t', encoding='latin1')
print(df.head())

Email_Data = df[['label', 'message']].rename(columns={"label": "Target", "message": "Email"})
print(Email_Data.head())

print()

Email_Data['Email'] = Email_Data['Email'].apply(preprocess_text)

print(Email_Data.head())
print()

  label                                            message  length  punct
0   ham  Go until jurong point, crazy.. Available only ...     111      9
1   ham                      Ok lar... Joking wif u oni...      29      6
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...     155      6
3   ham  U dun say so early hor... U c already then say...      49      6
4   ham  Nah I don't think he goes to usf, he lives aro...      61      2
  Target                                              Email
0    ham  Go until jurong point, crazy.. Available only ...
1    ham                      Ok lar... Joking wif u oni...
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...
3    ham  U dun say so early hor... U c already then say...
4    ham  Nah I don't think he goes to usf, he lives aro...

  Target                                              Email
0    ham  go jurong point crazy available bugis n great ...
1    ham                            ok lar joking wif u oni
2   spam  free 

In [5]:
# 3. Split data into train and test sets
train_x, test_x, train_y, test_y = train_test_split(
    Email_Data['Email'], # Features (input data)
    Email_Data['Target'],  # Labels (target data)
    test_size=0.2,  # 20% of the data is used for testing, and the remaining 80% is used for training.
    random_state=42, # If you set random_state=42, the split will be the same every time you rerun the code.
    shuffle= True # Whether to shuffle data before splitting
)

In [6]:
# 4. Create TF-IDF features
# analyser = 'word' The vectorizer splits text into individual words (tokens).
# \w = Matches alphanumeric characters + underscore ([a-zA-Z0-9_]).
# Meaning: Limits the vocabulary to the top 5,000 most frequent words in the corpus
tfidf_vect = TfidfVectorizer(analyzer='word', token_pattern=r'\w{1,}', max_features=5000)
# fit_transform(train_x)This is the learning and conversion phase:
# 1- Fit (Learning):Vocabulary: 
# It scans train_x and builds a dictionary of all unique words (tokens) based on your token_pattern=r'\w{1,}'.
# Max Features: It selects only the top 5,000 most frequent words because we  set max_features=5000.
# IDF Calculation: It calculates the Inverse Document Frequency (IDF) 
# 2-Transform (Converting):
#It uses that learned vocabulary and those IDF values to convert the raw text of train_x into the numerical matrix xtrain_tfidf
xtrain_tfidf = tfidf_vect.fit_transform(train_x)
# transform(test_x)This is the application-only phase for your test set. It does not learn anything new
# Uses Existing Rules: It converts test_x into a numerical matrix (xtest_tfidf) using only the 5,000 words and IDF values it learned from the training data.
#  the vectorizer calculates the TF (Term Frequency) from the new data, but it uses the IDF (Inverse Document Frequency) that was "frozen" from the training data.
xtest_tfidf = tfidf_vect.transform(test_x)

In [7]:
# 5. Train and evaluate models
'''
pos_label='spam'
Precision	TP / (TP + FP)	Only counts 'spam' as positive.
Recall	TP / (TP + FN)	Only considers 'spam' misses as false negatives.
F1-Score	2 * (Precision * Recall) / (Precision + Recall)	Harmonic mean of the two above.
'''
def train_model(classifier, X_train, y_train, X_test, y_test):
    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_test)
    return {
        'accuracy': metrics.accuracy_score(y_test, predictions),
        'precision': metrics.precision_score(y_test, predictions, pos_label='spam'),
        'recall': metrics.recall_score(y_test, predictions, pos_label='spam'),
        'f1': metrics.f1_score(y_test, predictions, pos_label='spam')
    }

# Initialize models
nb_model = naive_bayes.MultinomialNB(alpha=0.2)
lr_model = linear_model.LogisticRegression(max_iter=1000)

# Train and show results
'''
alpha is a smoothing parameter (also called Laplace smoothing or additive smoothing) 
that prevents zero probabilities in Naive Bayes when a word in the test data was not seen in training.
'''
nb_results = train_model(nb_model, xtrain_tfidf, train_y, xtest_tfidf, test_y)
'''
max_iter sets the maximum number of iterations for the solver (optimization algorithm) to converge.
'''
lr_results = train_model(lr_model, xtrain_tfidf, train_y, xtest_tfidf, test_y)

print("Naive Bayes Results:")
for metric, value in nb_results.items():
    print(f"{metric.capitalize()}: {value:.4f}")

print("\nLogistic Regression Results:")
for metric, value in lr_results.items():
    print(f"{metric.capitalize()}: {value:.4f}")

# 6. Save the best performing model
joblib.dump({
    'model': nb_model,  # Using Naive Bayes as example
    'vectorizer': tfidf_vect,
    'preprocessor': preprocess_text
}, 'spam_classifier.joblib')

print("\nModel saved to 'spam_classifier.joblib'")

Naive Bayes Results:
Accuracy: 0.9857
Precision: 1.0000
Recall: 0.8926
F1: 0.9433

Logistic Regression Results:
Accuracy: 0.9677
Precision: 0.9913
Recall: 0.7651
F1: 0.8636

Model saved to 'spam_classifier.joblib'


In [8]:
saved = joblib.load('spam_classifier.joblib')

# Example prediction
new_email = "Congratulations! You've won a free gift card!"
# Fetches your saved text preprocessing function/object and applies it to the raw email
processed = saved['preprocessor'](new_email)
# Converts the cleaned text string into a numerical feature vector.
vectorized = saved['vectorizer'].transform([processed])
prediction = saved['model'].predict(vectorized)[0]
print(f"Prediction: {prediction}")

Prediction: spam


In [9]:
new_email = "Hello dear friend can we meet tomorrow ?"
processed = saved['preprocessor'](new_email)
vectorized = saved['vectorizer'].transform([processed])
prediction = saved['model'].predict(vectorized)[0]
print(f"Prediction: {prediction}")

Prediction: ham
